[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/01_sft/01_sft_loss_masking.ipynb)

# 01 · SFT 与 loss masking — 动手实验

配套讲解：`01_讲解.html`（先读理论，再跑本 notebook）。

**本 notebook 内容**（默认全程 CPU 可跑，零下载也能跑通——真实 tokenizer 加载失败时自动回退）：

1. **chat template 实战**：用 `Qwen/Qwen2.5-0.5B-Instruct` 的真实 tokenizer（只下载 tokenizer，几 MB）观察 `apply_chat_template` 的输出与特殊 token；
2. **手写 loss masking**：对一条 (system, user, assistant) 对话构造 `input_ids` 与 `labels`，逐 token 打印 (id, label) 对照表；
3. **玩具验证**：固定小 logits 下对比 masked / unmasked loss 的数值与梯度去向；
4. **LoRA 从零实现**：纯 torch 写 `LoRALinear`，玩具回归上训练，验证 merge 等价；
5. **（可选重型）** peft + Qwen2.5-0.5B-Instruct 真实 LoRA SFT；
6. ✏️ 三道练习 + 📖 参考答案。


## 1 · chat template 实战：从 messages 到 token 流

模型只认 token 序列，`apply_chat_template` 负责把结构化的 messages 列表序列化成扁平文本/token 流。
下面尝试加载 Qwen 的真实 tokenizer（`from_pretrained` 只拉 tokenizer 文件，约几 MB）；
失败（无网络/未装 transformers）则回退到一个手写的 ChatML 风格迷你实现——格式与 Qwen 同构，后续所有 cell 不受影响。

重点观察两件事：
- **特殊 token**：`<|im_start|>`、`<|im_end|>` 是词表中的单个保留 token，`<|im_end|>` 兼任停止信号；
- **generation prompt**：推理时要在末尾追加 `<|im_start|>assistant\n`，模型才知道"该你说了"。


In [ ]:
import re
import torch
import torch.nn.functional as F

torch.manual_seed(0)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

MESSAGES = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "用一句话解释什么是 loss masking"},
    {"role": "assistant", "content": "只对 assistant 段的 token 算交叉熵，prompt 段 label 置 -100。"},
]

def mini_chat_template(messages, add_generation_prompt=False):
    # ChatML 风格（与 Qwen 模板同构）的最小实现
    s = ""
    for m in messages:
        s += "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    if add_generation_prompt:
        s += "<|im_start|>assistant\n"
    return s

class MiniTokenizer:
    # 确定性回退 tokenizer：特殊 token 整体成 token，其余逐字符；接口与 HF tokenizer 对齐
    PAT = re.compile(r"<\|im_start\|>|<\|im_end\|>|\n|\S| ")
    def __init__(self):
        self.vocab, self.inv = {}, {}
    def _id(self, t):
        if t not in self.vocab:
            self.vocab[t] = len(self.vocab)
            self.inv[self.vocab[t]] = t
        return self.vocab[t]
    def __call__(self, text, add_special_tokens=False):
        return {"input_ids": [self._id(t) for t in self.PAT.findall(text)]}
    def convert_ids_to_tokens(self, ids):
        return [self.inv[i] for i in ids]
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
        text = mini_chat_template(messages, add_generation_prompt)
        return self(text)["input_ids"] if tokenize else text

try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID)   # 只下载 tokenizer，几 MB
    USING_REAL = True
    print("✓ 已加载真实 tokenizer:", MODEL_ID)
except Exception as e:
    tok = MiniTokenizer()
    USING_REAL = False
    print(f"✗ 真实 tokenizer 不可用（{type(e).__name__}），回退到 MiniTokenizer，全部 cell 仍可运行")

print("\n=== 完整对话序列化（训练时的形态）===")
print(tok.apply_chat_template(MESSAGES, tokenize=False))

print("=== 只喂 system+user，add_generation_prompt=True（推理时的形态）===")
print(tok.apply_chat_template(MESSAGES[:-1], tokenize=False, add_generation_prompt=True))
print("↑ 注意结尾的 <|im_start|>assistant —— 没有它模型不会以 assistant 身份作答")

## 2 · 手写 loss masking：构造 input_ids 与 labels

定位 assistant 段边界的标准技巧（与 TRL `DataCollatorForCompletionOnlyLM` 思路一致）：

- **prompt 部分** = `apply_chat_template(system+user, add_generation_prompt=True)` —— 它把 `<|im_start|>assistant\n` 这个"回答前缀"也划进 prompt（属于模板、不该算 loss）；
- **完整序列** = `apply_chat_template(整段对话)`；
- 完整序列的 token 前缀就是 prompt，边界之后到 `<|im_end|>`（含！）才是要学的内容。

labels 规则：prompt 位置一律 `-100`，assistant 位置等于 `input_ids` 本身（HuggingFace 模型内部会自动右移一位对齐 logits）。


In [ ]:
prompt_text = tok.apply_chat_template(MESSAGES[:-1], tokenize=False, add_generation_prompt=True)
full_text   = tok.apply_chat_template(MESSAGES,      tokenize=False)
assert full_text.startswith(prompt_text), "template 不一致：prompt 不是完整序列的字符前缀！"

prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
full_ids   = tok(full_text,   add_special_tokens=False)["input_ids"]

# 字符级前缀不保证 token 级前缀（BPE 边界效应），稳妥做法：取最长公共 token 前缀
k = 0
while k < min(len(prompt_ids), len(full_ids)) and prompt_ids[k] == full_ids[k]:
    k += 1
print(f"prompt 段 = 前 {k} 个 token / 总长 {len(full_ids)}")

labels = [-100] * k + full_ids[k:]
assert len(labels) == len(full_ids)

toks = tok.convert_ids_to_tokens(full_ids)
print(f"\n{'idx':>4}  {'token':<24} {'id':>8} {'label':>8}")
print("-" * 50)
for i, (t, tid, lab) in enumerate(zip(toks, full_ids, labels)):
    mark = "   ← 算 loss" if lab != -100 else ""
    print(f"{i:>4}  {repr(t):<24.24} {tid:>8} {lab:>8}{mark}")

print("\n检查三个经典坑：")
print("① <|im_start|>assistant 回答前缀已被划入 prompt（label=-100）✓")
print("② 回答末尾的 <|im_end|> 保留 loss（模型要学会停）✓")
print("③ 多轮对话需对每个 assistant 段重复此操作（见练习 1）")

## 3 · 玩具验证：mask 切断的是反向，不是前向

用一个固定的小 logits 矩阵（序列长 5、词表 6，前 3 个位置是 prompt）直接对比：

- **数值**：masked loss 只平均后 2 个位置，与 unmasked（5 个位置平均）数值不同；
- **梯度去向**：对 masked loss 反传后，被 mask 位置的 logits 梯度**恰为零**——
  prompt 的信息仍通过 attention 影响前向，但它们自己不贡献任何梯度。


In [ ]:
torch.manual_seed(0)
V, T = 6, 5                                       # 词表 6，序列 5；前 3 个位置是 prompt
targets = torch.tensor([2, 4, 1, 3, 5])           # 不 mask：所有位置都当 label
masked_labels = torch.tensor([-100, -100, -100, 3, 5])  # mask：prompt 置 -100

logits_a = torch.randn(T, V, requires_grad=True)
logits_b = logits_a.detach().clone().requires_grad_(True)   # 同一份 logits 两份拷贝

loss_unmasked = F.cross_entropy(logits_a, targets)
loss_masked   = F.cross_entropy(logits_b, masked_labels, ignore_index=-100)
print(f"unmasked loss = {loss_unmasked.item():.4f}   (5 个位置平均)")
print(f"masked   loss = {loss_masked.item():.4f}   (只平均后 2 个位置)")

loss_unmasked.backward()
loss_masked.backward()

print("\n各位置梯度范数（行 = 序列位置）：")
print(f"{'pos':>4} {'unmasked |grad|':>16} {'masked |grad|':>14}")
for t in range(T):
    print(f"{t:>4} {logits_a.grad[t].norm().item():>16.4f} {logits_b.grad[t].norm().item():>14.4f}")

assert torch.allclose(logits_b.grad[:3], torch.zeros(3, V)), "被 mask 的位置不应有梯度"
assert logits_b.grad[3:].abs().sum() > 0
print("\n✓ masked：前 3 行（prompt）梯度恰为 0，梯度全部流向 assistant 段")
print("✓ unmasked：每一行都有梯度 —— prompt 越长，回答信号被稀释得越狠")

## 4 · LoRA 从零实现：`LoRALinear`

$$h = W_0 x + \frac{\alpha}{r} BA\,x,\qquad A\in\mathbb{R}^{r\times d_{in}},\ B\in\mathbb{R}^{d_{out}\times r}$$

纯 torch 实现并验证三件事 [Hu 2021]：

1. **零扰动起步**：$B=0$ 初始化 ⇒ 训练开始时前向与原模型严格一致；
2. **只训低秩分支**：$W_0$ 冻结，可训练参数占比极小，但玩具回归 loss 照样下降；
3. **可合并推理**：$W' = W_0 + \frac{\alpha}{r}BA$ 后，普通 Linear 前向与 LoRA 分支前向逐元素等价 ⇒ 零额外延迟。


In [ ]:
class LoRALinear(torch.nn.Module):
    # y = W0 x + (alpha/r) * B(Ax)；W0 冻结，只训练 A、B
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.r, self.alpha = r, alpha
        self.A = torch.nn.Parameter(torch.randn(r, base.in_features) * 0.02)  # 随机小高斯
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))        # 零初始化！
    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * F.linear(F.linear(x, self.A), self.B)
    def merged_weight(self):
        return self.base.weight.data + (self.alpha / self.r) * (self.B @ self.A)

torch.manual_seed(42)
d_in, d_out, n, r_true = 128, 64, 256, 4
base = torch.nn.Linear(d_in, d_out, bias=False)

# 构造任务：目标权重 = base 权重 + 一个真实低秩偏移（正是 LoRA 的核心假设：微调的 ΔW 本征秩低）
delta_true = (torch.randn(d_out, r_true) @ torch.randn(r_true, d_in)) / d_in ** 0.5
W_teacher = base.weight.data + delta_true
X = torch.randn(n, d_in)
Y = X @ W_teacher.T + 0.01 * torch.randn(n, d_out)   # 玩具回归任务

lora = LoRALinear(base, r=4, alpha=8)
W0_snapshot = base.weight.data.clone()

# ① B=0 ⇒ 初始前向与 base 严格一致
assert torch.allclose(lora(X), base(X))
print("✓ 初始前向 == base 前向（零扰动起步）")

trainable = [p for p in lora.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_total = sum(p.numel() for p in lora.parameters())
print(f"可训练参数: {n_train}/{n_total} = {100 * n_train / n_total:.1f}%  (只有 A 和 B；真实 LLM 上通常 <1%)")

# ② 训几十步看 loss 下降（r=4 >= 真实秩，足以拟合低秩偏移）
opt = torch.optim.Adam(trainable, lr=2e-2)
for step in range(80):
    loss = F.mse_loss(lora(X), Y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 20 == 0 or step == 79:
        print(f"step {step:>3}  mse = {loss.item():.5f}")

assert torch.equal(base.weight.data, W0_snapshot), "W0 应保持冻结不变"
print("✓ 训练全程 W0 未动")

# ③ merge 后与 LoRA 分支前向等价
y_branch = lora(X)
y_merged = X @ lora.merged_weight().T
assert torch.allclose(y_branch, y_merged, atol=1e-5)
print("✓ merge 后前向与 LoRA 分支逐元素等价 —— 推理零额外延迟")

## 5 · （可选重型）peft + Qwen2.5-0.5B-Instruct 真实 LoRA SFT

> ⚠️ **资源标注**：首次运行约 **1 GB 模型下载**；纯 CPU 上每步约数十秒，3 步约 2–5 分钟。
> 需要 `pip install transformers peft`。默认 `RUN_HEAVY = False` 跳过——上面的纯 torch 实现已覆盖全部原理，
> 这个 cell 只为看一眼真实规模下的 trainable% 与 loss 下降。

流程与第 2 节完全相同：`apply_chat_template` 构造 full/prompt → prompt 位置 `-100` → `model(input_ids, labels)`（HF 内部自动 shift）。


In [ ]:
RUN_HEAVY = False   # ← 改为 True 才执行（约 1GB 下载，CPU 慢）

if not RUN_HEAVY:
    print("跳过重型 cell（RUN_HEAVY=False）")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model

    tok_h = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                     task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()   # 预期 ~0.2% trainable

    SAMPLES = [
        ("什么是 loss masking？", "只对 assistant 段计算交叉熵，prompt 段 label 置 -100。"),
        ("LoRA 的 B 为什么零初始化？", "让训练开始时 ΔW=BA=0，前向与原模型一致。"),
        ("SFT 的本质是什么？", "对人类示范的行为克隆，损失仍是 next-token prediction。"),
        ("chat template 不一致会怎样？", "训练与推理的 token 模式错位，输出格式与质量崩坏。"),
    ]

    def encode_sample(q, a):
        msgs = [{"role": "user", "content": q}]
        prompt = tok_h.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        full   = tok_h.apply_chat_template(
            msgs + [{"role": "assistant", "content": a}], tokenize=False)
        ids  = tok_h(full, add_special_tokens=False, return_tensors="pt").input_ids
        plen = len(tok_h(prompt, add_special_tokens=False)["input_ids"])
        lab  = ids.clone()
        lab[0, :plen] = -100          # 与第 2 节完全相同的 masking
        return ids, lab

    opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-4)
    model.train()
    for step in range(3):
        total = 0.0
        for q, a in SAMPLES:
            ids, lab = encode_sample(q, a)
            loss = model(input_ids=ids, labels=lab).loss
            loss.backward()
            total += loss.item()
        opt.step()
        opt.zero_grad()
        print(f"step {step}  mean loss = {total / len(SAMPLES):.4f}")

## ✏️ 练习 1：多轮对话的 `build_labels`

多轮对话里有多个 assistant 段，**每一段都要算 loss**，其余（system/user/role 标记）全部 `-100`。

实现 `build_labels(input_ids, assistant_spans)`：

- `input_ids: List[int]`；
- `assistant_spans: List[Tuple[int, int]]`，每个 `(start, end)` 是一个 assistant 段，**左闭右开**；
- 返回 `labels: List[int]`：span 内 `labels[i] = input_ids[i]`，span 外 `-100`。

提示：先全部初始化为 `-100`，再逐 span 回填。约 5 行。


In [ ]:
def build_labels(input_ids, assistant_spans):
    # TODO: 初始化全 -100，再把每个 (start, end) 左闭右开区间内回填为 input_ids[i]
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ids = list(range(100, 112))                       # 12 个 token
out = build_labels(ids, [(3, 6), (9, 12)])
assert out == [-100, -100, -100, 103, 104, 105, -100, -100, -100, 109, 110, 111]

# 多轮对话：两个 assistant 段都保留，其余 -100
multi = build_labels(list(range(10)), [(2, 4), (7, 9)])
assert multi == [-100, -100, 2, 3, -100, -100, -100, 7, 8, -100]

# 边界：无 assistant 段 → 全 -100
assert build_labels(ids, []) == [-100] * 12
# 边界：span 覆盖整个序列
assert build_labels([7, 8], [(0, 2)]) == [7, 8]
print("✅ 练习 1 通过")

## ✏️ 练习 2：手写 `masked_ce_loss`

不借助 `ignore_index`，自己实现忽略 `-100` 的平均交叉熵，并与
`F.cross_entropy(logits, labels, ignore_index=-100)` 对拍。

`masked_ce_loss(logits, labels)`：`logits` 形状 `(T, V)`，`labels` 形状 `(T,)` 含 `-100`。

提示（约 6 行）：
1. `mask = labels != -100`；
2. `logp = F.log_softmax(logits, dim=-1)`；
3. 把 `-100` 先 `clamp(min=0)` 成合法下标，`gather` 出每个位置的 log prob；
4. 取负、乘 mask、除以 `mask.sum()`。


In [ ]:
def masked_ce_loss(logits, labels):
    # TODO: log_softmax + gather 实现，仅对 labels != -100 的位置求平均交叉熵
    # 注意 -100 不是合法的类下标，gather 前要先 clamp
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测：与 F.cross_entropy(ignore_index=-100) 对拍 ——
torch.manual_seed(1)
for trial in range(3):
    T, V = 8, 10
    lg = torch.randn(T, V)
    lb = torch.randint(0, V, (T,))
    lb[: 3 * trial] = -100                        # trial=0 不 mask；1/2 部分 mask
    ref = F.cross_entropy(lg, lb, ignore_index=-100)
    got = masked_ce_loss(lg, lb)
    assert torch.allclose(got, ref, atol=1e-6), f"trial {trial}: {got.item()} vs {ref.item()}"

# 边界：只剩 1 个有效位置
lg = torch.randn(5, 10)
lb = torch.full((5,), -100, dtype=torch.long)
lb[2] = 4
assert torch.allclose(masked_ce_loss(lg, lb),
                      F.cross_entropy(lg, lb, ignore_index=-100), atol=1e-6)
print("✅ 练习 2 通过")

## ✏️ 练习 3：`lora_merge`

实现 LoRA 的权重合并：

$$W' = W + \frac{\alpha}{r}\,B A$$

`lora_merge(W, A, B, alpha, r)`：`W` 形状 `(d_out, d_in)`、`A` 形状 `(r, d_in)`、`B` 形状 `(d_out, r)`，
返回合并后的稠密权重（不修改原 `W`）。1–2 行即可。

自测验证：合并后的普通矩阵乘前向 == 原 `W` 前向 + LoRA 分支前向。


In [ ]:
def lora_merge(W, A, B, alpha, r):
    # TODO: 返回 W + (alpha/r) * B @ A
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测：合并前后前向等价 ——
torch.manual_seed(2)
d_in, d_out, r, alpha = 6, 4, 2, 8
W = torch.randn(d_out, d_in)
A = torch.randn(r, d_in)
B = torch.randn(d_out, r)
x = torch.randn(3, d_in)

W_m = lora_merge(W, A, B, alpha, r)
assert W_m.shape == W.shape

y_branch = x @ W.T + (alpha / r) * (x @ A.T @ B.T)   # 分支式前向
y_merged = x @ W_m.T                                  # 合并后前向
assert torch.allclose(y_branch, y_merged, atol=1e-6)

# 边界：B = 0 ⇒ 合并结果就是原 W
assert torch.equal(lora_merge(W, A, torch.zeros(d_out, r), alpha, r), W)
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。运行下面任一答案 cell 后，回到对应自测 cell 重新运行即可验证。


In [ ]:
# 参考答案 · 练习 1（先自己做，再对照）
def build_labels(input_ids, assistant_spans):
    labels = [-100] * len(input_ids)
    for start, end in assistant_spans:        # 左闭右开
        for i in range(start, end):
            labels[i] = input_ids[i]
    return labels

In [ ]:
# 参考答案 · 练习 2（先自己做，再对照）
def masked_ce_loss(logits, labels):
    mask = labels != -100
    logp = F.log_softmax(logits, dim=-1)
    safe = labels.clamp(min=0)                          # -100 不是合法下标，先钳到 0
    nll = -logp.gather(-1, safe.unsqueeze(-1)).squeeze(-1)
    return (nll * mask).sum() / mask.sum()

In [ ]:
# 参考答案 · 练习 3（先自己做，再对照）
def lora_merge(W, A, B, alpha, r):
    return W + (alpha / r) * (B @ A)

## 小结

- **SFT = 行为克隆**：损失仍是 next-token prediction，变的是数据形态（chat template 序列化的对话）与 **loss masking**（prompt 段 label=-100，只学"如何回答"）；
- mask 切断的是**反向**不是前向：prompt 信息照常通过 attention 流入，但梯度只来自 assistant 段（含结尾的 `<|im_end|>`——学会停止）；
- **LoRA**：$W_0$ 冻结 + $\frac{\alpha}{r}BA$ 低秩分支，$B$ 零初始化保证零扰动起步，训练后可合并 ⇒ 推理零额外延迟；
- 工程红线：训练/推理 **chat template 必须逐 token 一致**；loss 下降 ≠ 对话变好。

**下一步 → 模块 02 · 奖励模型**：SFT 只有"模仿"没有"比较"——它无法告诉模型两个回答哪个更好。
Bradley-Terry 偏好建模如何把人类比较变成标量奖励、以及 reward hacking 为什么不可避免，见 `../02_reward_models/02_讲解.html`。


---
## 🎯 真实数据胶囊题：真实 GSM8K 上的 SFT loss masking

SFT 只在**回答**部分算 loss，prompt 部分要 mask 掉。下载真实 GSM8K（小学数学题），把 question 当 prompt、answer 当 completion，构造 loss mask，验证 prompt token 不贡献 loss。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

ex = gsm8k(50)[0]
prompt = ex["question"]; completion = ex["answer"]
# 字符级 token（演示用）：prompt + completion 拼接
prompt_ids = list(prompt); comp_ids = list(completion)
total = prompt_ids + comp_ids
print(f"真实题目: {prompt[:60]}...")
print(f"prompt {len(prompt_ids)} tokens, completion {len(comp_ids)} tokens")

**练习**：实现 `build_loss_mask(n_prompt, n_total)`（prompt 位置 0、completion 位置 1）和 `masked_loss(per_token_loss, mask)`（只对 mask=1 的位置求平均）。

In [ ]:
def build_loss_mask(n_prompt, n_total):
    # TODO: 前 n_prompt 个为 0，其余为 1 的数组
    raise NotImplementedError
def masked_loss(per_token_loss, mask):
    # TODO: sum(loss*mask)/sum(mask)
    raise NotImplementedError


In [ ]:
# 自测
n_p, n_t = len(prompt_ids), len(total)
mask = build_loss_mask(n_p, n_t)
assert mask[:n_p].sum()==0 and mask[n_p:].sum()==(n_t-n_p), "prompt 全 mask, completion 全计"
rng=np.random.default_rng(0); losses=rng.random(n_t)
full = losses.mean(); ml = masked_loss(losses, mask)
assert abs(ml - losses[n_p:].mean()) < 1e-9, "masked loss 只看 completion"
assert ml != full, "和全 token 平均不同"
print(f"SFT masking ✓  completion-only loss={ml:.3f} (vs 全token {full:.3f})")


### 📖 参考答案

In [ ]:
def build_loss_mask(n_prompt, n_total):
    m=np.ones(n_total); m[:n_prompt]=0; return m
def masked_loss(per_token_loss, mask):
    return float((per_token_loss*mask).sum()/mask.sum())
print("✓ 不 mask prompt，模型会去'背题目'而非学'答题'")